In [ ]:
import os
import shutil
import time
import subprocess
import sys
import re
from datetime import datetime
from pathlib import Path

In [ ]:
# --- Setup & Dependencies ---
def run_bootstrap_command(cmd, label):
    print(f"[SETUP] {label}")
    subprocess.run(cmd, check=True)

def ensure_system_tools():
    missing = [tool for tool in ["ffmpeg", "node"] if shutil.which(tool) is None]
    if missing:
        run_bootstrap_command(["apt-get", "update", "-qq"], "Preparing Colab package index")
        run_bootstrap_command(["apt-get", "install", "-y", "ffmpeg", "nodejs"], "Installing ffmpeg & nodejs")

def ensure_python_package(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        __import__(import_name)
    except ModuleNotFoundError:
        run_bootstrap_command([sys.executable, "-m", "pip", "install", "-q", package_name], "Installing " + package_name)

ensure_system_tools()
ensure_python_package("yt_dlp", "yt-dlp")
ensure_python_package("ipywidgets")

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive
from yt_dlp import YoutubeDL

In [ ]:
# --- Core Application ---
class YouTubeSpecialistDL:
    def __init__(self):
        self.drive_mount_path = Path("/content/drive")
        self.my_drive_path = self.drive_mount_path / "MyDrive"
        self.youtube_base_dir = self.my_drive_path / "YouTubeDownloads"
        self.temp_dir = Path("/content/YT-Temp")

    def mount_google_drive(self):
        if not self.my_drive_path.exists():
            print("Mounting Google Drive...")
            drive.mount(str(self.drive_mount_path))
        else:
            print("Google Drive is already mounted.")
        self.youtube_base_dir.mkdir(parents=True, exist_ok=True)

    def sanitize_path(self, raw_path):
        raw_path = str(raw_path or "").strip()
        if not raw_path:
            return Path("MyPlaylist")

        parts = [re.sub(r'[*?"<>|:]', "_", p.strip()) for p in re.split(r'[/\\]', raw_path) if p.strip()]
        return Path(*parts) if parts else Path("MyPlaylist")

    def force_drive_sync(self):
        try:
            subprocess.run(["sync"], check=False)
        except Exception:
            pass
        time.sleep(2)

    def is_temporary_output_file(self, path):
        path = Path(path)
        name = path.name.lower()
        suffix = path.suffix.lower()
        if suffix in [".part", ".ytdl", ".temp", ".tmp"]: return True
        if name.endswith(".part") or name.endswith(".ytdl") or name.endswith(".tmp"): return True
        return False

    def collect_final_files(self, folder):
        folder = Path(folder)
        if not folder.exists(): return []
        files = []
        for item in folder.glob("**/*"):
            if item.is_file() and not self.is_temporary_output_file(item):
                try:
                    if item.stat().st_size > 0: files.append(item)
                except Exception:
                    pass
        return files

    def transfer_folder_contents_to_drive(self, source_dir, destination_dir):
        source_dir = Path(source_dir)
        destination_dir = Path(destination_dir)
        destination_dir.mkdir(parents=True, exist_ok=True)

        files = self.collect_final_files(source_dir)
        transferred = []

        for item in files:
            target = destination_dir / item.name
            counter = 1
            while target.exists():
                target = destination_dir / f"{item.stem}_{counter}{item.suffix}"
                counter += 1

            shutil.copy2(item, target)
            transferred.append(target)
            item.unlink()

        self.force_drive_sync()
        return transferred

    def get_existing_folders(self):
        folders = ["-- Select Existing Folder --"]
        if self.youtube_base_dir.exists():
            for p in sorted(self.youtube_base_dir.glob("**/*")):
                if p.is_dir():
                    rel = p.relative_to(self.youtube_base_dir)
                    folders.append(str(rel))
        return folders

    def get_format_config(self, dl_type, quality):
        res_limit = "" if quality == "Best" else f"[height<={quality.replace('p', '')}]"

        if dl_type == "Audio Only":
            format_str = "bestaudio/best"
            postprocessors = [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }]
        elif dl_type == "Video Only":
            format_str = f"bestvideo{res_limit}[ext=mp4]/bestvideo{res_limit}/bestvideo"
            postprocessors = []
        else:  # Video + Audio
            format_str = f"bestvideo{res_limit}[ext=mp4]+bestaudio[ext=m4a]/best{res_limit}[ext=mp4]/best{res_limit}/best"
            postprocessors = []

        return format_str, postprocessors

    def extract_downloaded_ids_from_archive(self, archive_file):
        if not archive_file.exists():
            return set()

        archive_ids = set()
        for line in archive_file.read_text().splitlines():
            line = line.strip()
            if line:
                parts = line.split()
                if len(parts) >= 2:
                    archive_ids.add(parts[1])
                elif len(parts) == 1:
                    archive_ids.add(parts[0])
        return archive_ids

    def record_history_entry(self, history_file, video_id, title, url):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        entry = f"[{timestamp}] ID: {video_id} | Title: {title} | Link: {url}\n"
        with open(history_file, "a", encoding="utf-8") as f:
            f.write(entry)

    def download_single_entry(self, entry, target_dir, archive_file, history_file, dl_type, quality, want_subs, sub_language, add_index):
        """Attempts to download a single entry. Returns True on success/already done, False on failure."""
        vid_id = entry.get('id')
        vid_title = entry.get('title') or "Unknown Title"
        vid_url = entry.get('url')
        orig_idx = entry.get('_orig_idx', 1)

        if not vid_url and vid_id:
            vid_url = f"https://www.youtube.com/watch?v={vid_id}"

        if not vid_url:
            return True

        format_str, postprocessors = self.get_format_config(dl_type, quality)

        # Determine output filename template with index prefix if requested
        filename_prefix = f"{orig_idx}-" if add_index else ""
        out_template = str(self.temp_dir / f"{filename_prefix}%(title).190s.%(ext)s")

        ydl_dl_opts = {
            'format': format_str,
            'outtmpl': out_template,
            'download_archive': str(archive_file),
            'postprocessors': postprocessors,
            'quiet': False,
            'noprogress': False,
            'ignoreerrors': 'only_download',
            'sleep_interval': 2,
            'max_sleep_interval': 4,
        }

        if dl_type == "Video + Audio":
            ydl_dl_opts['merge_output_format'] = 'mp4'

        if want_subs:
            ydl_dl_opts['writesubtitles'] = True
            ydl_dl_opts['writeautomaticsub'] = True
            ydl_dl_opts['subtitleslangs'] = [sub_language, 'en']
            ydl_dl_opts['subtitlesformat'] = 'srt/best'

        try:
            with YoutubeDL(ydl_dl_opts) as ydl:
                download_result = ydl.extract_info(vid_url, download=True)
                if download_result:
                    vid_title = download_result.get('title', vid_title)
                    vid_id = download_result.get('id', vid_id)
        except Exception as e:
            print(f"[ERROR] Exception during download for {vid_url}: {e}")
            return False

        transferred = self.transfer_folder_contents_to_drive(self.temp_dir, target_dir)
        if transferred:
            print(f"[SUCCESS] Moved {len(transferred)} file(s) to Google Drive.")
            if vid_id:
                self.record_history_entry(history_file, vid_id, vid_title, vid_url)
            return True
        else:
            completed_ids = self.extract_downloaded_ids_from_archive(archive_file)
            if vid_id and vid_id in completed_ids:
                print("[INFO] Item skipped (already recorded in archive).")
                return True
            else:
                print("[WARNING] Download produced no output files.")
                return False

    def render_ui(self):
        print("=== YouTube Specialist Downloader ===")
        self.mount_google_drive()

        existing_folders = self.get_existing_folders()

        self.url_input = widgets.Text(
            description="YouTube URL(s):",
            placeholder="Paste URL(s) here. Separate multiple links with '+' (e.g., link1 + link2)",
            layout=widgets.Layout(width='90%')
        )

        self.folder_dropdown = widgets.Dropdown(
            options=existing_folders,
            value=existing_folders[0],
            description="Existing Dir:",
            layout=widgets.Layout(width='90%')
        )

        self.folder_text = widgets.Text(
            description="Or New Path:",
            placeholder="Enter folder path (e.g., CourseName/Module1)",
            layout=widgets.Layout(width='90%')
        )

        self.dl_type = widgets.Dropdown(
            options=["Video + Audio", "Audio Only", "Video Only"],
            value="Video + Audio",
            description="Download Type:",
            layout=widgets.Layout(width='90%')
        )

        self.quality = widgets.Dropdown(
            options=["Best", "1080p", "720p", "480p", "360p"],
            value="Best",
            description="Max Quality:",
            layout=widgets.Layout(width='90%')
        )

        self.download_subs = widgets.Checkbox(
            value=False,
            description="Download Subtitles",
            indent=False,
            layout=widgets.Layout(width='40%')
        )

        self.sub_lang = widgets.Text(
            value="en",
            description="Sub Lang:",
            placeholder="e.g., en, es, fr, ar, de",
            layout=widgets.Layout(width='48%')
        )

        self.subs_box = widgets.HBox([self.download_subs, self.sub_lang], layout=widgets.Layout(width='90%'))

        self.retry_count = widgets.Dropdown(
            options=[1, 2, 3, 5],
            value=3,
            description="Retry Passes:",
            layout=widgets.Layout(width='30%')
        )

        self.add_index = widgets.Checkbox(
            value=True,
            description="Add Video Index (e.g., 1-title)",
            indent=False,
            layout=widgets.Layout(width='35%')
        )

        self.fast_resume = widgets.Checkbox(
            value=False,
            description="Fast Resume",
            indent=False,
            layout=widgets.Layout(width='25%')
        )

        self.opts_box = widgets.HBox([self.retry_count, self.add_index, self.fast_resume], layout=widgets.Layout(width='90%'))

        self.start_btn = widgets.Button(description="Start Download", button_style='success', layout=widgets.Layout(width='90%'))
        self.start_btn.on_click(self.on_start)

        self.output = widgets.Output()

        display(
            self.url_input,
            self.folder_dropdown,
            self.folder_text,
            self.dl_type,
            self.quality,
            self.subs_box,
            self.opts_box,
            self.start_btn,
            self.output
        )

    def on_start(self, b):
        with self.output:
            clear_output()
            raw_url_input = self.url_input.value.strip()

            chosen_text = self.folder_text.value.strip()
            chosen_dropdown = self.folder_dropdown.value

            if chosen_text:
                folder_rel_path = self.sanitize_path(chosen_text)
            elif chosen_dropdown and chosen_dropdown != "-- Select Existing Folder --":
                folder_rel_path = Path(chosen_dropdown)
            else:
                folder_rel_path = Path("MyPlaylist")

            dl_type = self.dl_type.value
            quality = self.quality.value
            use_fast_resume = self.fast_resume.value
            want_subs = self.download_subs.value
            sub_language = self.sub_lang.value.strip() or "en"
            max_retry_passes = self.retry_count.value
            add_index = self.add_index.value

            if not raw_url_input:
                print("[ERROR] Please provide a valid YouTube URL.")
                return

            target_dir = self.youtube_base_dir / folder_rel_path
            target_dir.mkdir(parents=True, exist_ok=True)
            print(f"[INFO] Destination Directory: Google Drive -> MyDrive/YouTubeDownloads/{folder_rel_path}")

            archive_file = target_dir / "download_archive.txt"
            history_file = target_dir / "download_history.txt"

            if archive_file.exists():
                print(f"[INFO] Auto-detected existing 'download_archive.txt' in Drive ({folder_rel_path}). Resuming progress.")
            else:
                print(f"[INFO] Creating new 'download_archive.txt' in Drive ({folder_rel_path}).")
                archive_file.write_text("")

            self.temp_dir.mkdir(parents=True, exist_ok=True)

            url_list = [u.strip() for u in raw_url_input.split("+") if u.strip()]
            print(f"[INFO] Found {len(url_list)} URL target(s) separated by '+'. Processing metadata...")

            ydl_extract_opts = {'extract_flat': True, 'quiet': True}
            entries = []

            for u_idx, raw_url in enumerate(url_list, 1):
                try:
                    with YoutubeDL(ydl_extract_opts) as ydl:
                        info = ydl.extract_info(raw_url, download=False)
                        if 'entries' in info:
                            extracted = list(info['entries'])
                            print(f"[INFO] Target {u_idx}: Playlist containing {len(extracted)} item(s).")
                            entries.extend(extracted)
                        else:
                            print(f"[INFO] Target {u_idx}: Single Video.")
                            entries.append(info)
                except Exception as e:
                    print(f"[ERROR] Failed to extract metadata for '{raw_url}': {e}")

            if not entries:
                print("[ERROR] No valid entries found to download.")
                return

            # Attach index tracking to maintain original playlist position
            for idx, entry in enumerate(entries, 1):
                entry['_orig_idx'] = idx

            if use_fast_resume:
                completed_ids = self.extract_downloaded_ids_from_archive(archive_file)
                if completed_ids:
                    last_completed_idx = -1
                    for idx, entry in enumerate(entries):
                        v_id = entry.get('id') or (entry.get('url', '').split('v=')[-1] if 'v=' in entry.get('url', '') else None)
                        if v_id and v_id in completed_ids:
                            last_completed_idx = idx

                    if last_completed_idx != -1:
                        skipped_count = last_completed_idx + 1
                        entries = entries[skipped_count:]
                        print(f"[FAST RESUME] Found last completed video at index {skipped_count}. Skipping {skipped_count} item(s) instantly!")
                    else:
                        print("[FAST RESUME] No matching completed IDs found in archive. Starting from beginning.")
                else:
                    print("[FAST RESUME] Archive is empty. Starting from beginning.")

            total_remaining = len(entries)
            if total_remaining == 0:
                print("\n[DONE] All videos in the provided link(s) are already completed!")
                return

            # --- Primary Download Loop ---
            failed_entries = []
            print(f"\n--- Starting Primary Download Loop ({total_remaining} items) ---")

            for i, entry in enumerate(entries, 1):
                vid_title = entry.get('title') or "Unknown Title"
                vid_url = entry.get('url') or (f"https://www.youtube.com/watch?v={entry.get('id')}" if entry.get('id') else "")
                orig_idx = entry.get('_orig_idx', i)

                print(f"\n[{i}/{total_remaining}] Downloading (#{orig_idx} - {dl_type} @ {quality}): {vid_title}")
                print(f"      URL: {vid_url}")

                success = self.download_single_entry(
                    entry, target_dir, archive_file, history_file,
                    dl_type, quality, want_subs, sub_language, add_index
                )

                if not success:
                    failed_entries.append(entry)

            # --- Post-Loop Retry Mechanism ---
            if failed_entries:
                print(f"\n" + "=" * 60)
                print(f"[RETRY MECHANISM] Primary pass finished. {len(failed_entries)} item(s) failed.")
                print(f"=" * 60)

                for pass_num in range(1, max_retry_passes + 1):
                    if not failed_entries:
                        break

                    print(f"\n>>> RETRY PASS {pass_num}/{max_retry_passes} ({len(failed_entries)} item(s) remaining) <<<")
                    print("[INFO] Pausing 6 seconds to let rate-limits settle...")
                    time.sleep(6)

                    still_failed = []
                    for r_idx, entry in enumerate(failed_entries, 1):
                        vid_title = entry.get('title') or "Unknown Title"
                        vid_url = entry.get('url') or (f"https://www.youtube.com/watch?v={entry.get('id')}" if entry.get('id') else "")
                        orig_idx = entry.get('_orig_idx', r_idx)

                        print(f"\n[Retry {pass_num} - {r_idx}/{len(failed_entries)}] Retrying (#{orig_idx}): {vid_title}")
                        print(f"      URL: {vid_url}")

                        retry_success = self.download_single_entry(
                            entry, target_dir, archive_file, history_file,
                            dl_type, quality, want_subs, sub_language, add_index
                        )

                        if not retry_success:
                            still_failed.append(entry)

                    failed_entries = still_failed

            # Final Summary
            if failed_entries:
                print(f"\n[FINISHED] Execution completed with {len(failed_entries)} permanently failed item(s).")
            else:
                print("\n[DONE] Execution completed successfully! All items downloaded.")

            self.folder_dropdown.options = self.get_existing_folders()

In [ ]:
# Run UI
app = YouTubeSpecialistDL()
app.render_ui()